# Predictive accuracy: complete observations

Manuscript Section 6.2 / Table 3.

Use the separately fitted complete-case models, **not** full-data fits evaluated on fewer rows. All five models use the same 1e-6 diagonal jitter and no PSD correction.

The full evaluation contains **41 events and 49,995 observed residuals**. Parameters stay fixed. Entry and station holdouts mask 10% and use 20 seed-0 repetitions; period holdouts mask each of the nine periods once. Every model sees the same masks.

The notebook contains the recorded all-event results from the supplied fit, an executed whole-event demonstration, and the command for rerunning the complete evaluation. It writes no result files unless an `output` directory is explicitly passed to `evaluate`.


In [1]:
from pathlib import Path
import sys
import hashlib
from io import StringIO
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/models').is_dir())
sys.path.insert(0, str(ROOT))
from scripts.analysis.predictive_accuracy.evaluate import evaluate, smallest_event
pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 6)

DATASET = 'complete'
CACHE = ROOT / 'results/analysis/model_fit/models_complete.pkl'

## Recorded all-event evaluation

These values were computed from the supplied model file in the completed evaluation. The hash identifies that fit. RMSE and CRPS in the main table pool squared errors, CRPS sums and observation counts across the held-out residuals. The following cell also shows per-repeat/period means and win counts, making the aggregation explicit.

The original completed evaluation used the recorded-file hash below. The distributed complete-case PKL was subsequently reduced to its five fitted variants and its auxiliary plotting curves regenerated from those parameters. **All native fitted parameters and data arrays are unchanged.** Predictive evaluation uses those native parameters and never reads the auxiliary plotting curves. Both hashes are displayed to distinguish the original evaluation file from the cleaned distribution.

In [2]:
# Recorded all-event evaluation, 11 September 2026, before reorganizing the repository.
# These CSV strings preserve the completed numerical results inside the notebook.
# They are not results of the short demonstration below.
RECORDED_MODELS_SHA256 = '49a2a49147ad808ea375805a032d58a20bee5aa84a94fc45d1bbfb5c19609ec2'
DISTRIBUTED_MODELS_SHA256 = '773a73711e8112e28c7d0d05111def6c44456325906cda833a7d87d077a2e9ca'
recorded_summary = pd.read_csv(StringIO('method,wls,log_likelihood,pseudo_rmse,pseudo_crps,station_rmse,station_crps,period_rmse,period_crps\r\nKronecker semivariogram,0.2301548488466522,-27347.17059643979,0.3715435736503363,0.183785585310497,0.7991037727103977,0.4341778853940932,0.4255941559822776,0.2060858939099696\r\nPCA MLE,0.3643845836036936,-25005.973823905533,0.3680590457785446,0.1816741506826058,0.7947770424462054,0.4319299849709981,0.422420724297643,0.204499031692815\r\nPCA semivariogram,0.1990455199671532,-25992.745889039703,0.3706982608445444,0.1834893322559268,0.797753280616956,0.4333432691704048,0.4281820823170073,0.2078260409981498\r\nLMC block MLE,0.1604209646198497,-24260.028170630743,0.3616418272011174,0.1780699913473852,0.7923389391157579,0.429744381239153,0.4186421116646379,0.2028995262323752\r\nLMC semivariogram,0.094247525632246,-25248.894036001337,0.3661626264424024,0.1812109748036319,0.7941410358858703,0.4306890322240187,0.4249626994207684,0.2084932353522748\r\n'))
recorded_statistics = pd.read_csv(StringIO('method,task,metric,mean,sd,se,wins,mean_rank,n\r\nKronecker semivariogram,random_holdout,rmse,0.3715220910973776,0.0040991587705385,0.0009165997661488,0,4.75,20\r\nLMC block MLE,random_holdout,rmse,0.3616174149787173,0.004311107498816,0.0009639929425661,20,1.0,20\r\nLMC semivariogram,random_holdout,rmse,0.3661347351159195,0.0046367802796535,0.0010368155902035,0,2.15,20\r\nPCA MLE,random_holdout,rmse,0.3680362561725127,0.0042021701077038,0.0009396338013843,0,2.85,20\r\nPCA semivariogram,random_holdout,rmse,0.3706745018772996,0.0043059629628599,0.0009628425893551,0,4.25,20\r\nKronecker semivariogram,random_holdout,crps,0.183785585310497,0.0021413746817543,0.0004788259353699,0,4.7,20\r\nLMC block MLE,random_holdout,crps,0.1780699913473852,0.0020610227641823,0.0004608587003886,20,1.0,20\r\nLMC semivariogram,random_holdout,crps,0.1812109748036319,0.0021100173724935,0.0004718142278601,0,2.2,20\r\nPCA MLE,random_holdout,crps,0.1816741506826058,0.0020669944716609,0.000462194014775,0,2.8,20\r\nPCA semivariogram,random_holdout,crps,0.1834893322559268,0.0021260260932841,0.0004753938866521,0,4.3,20\r\nKronecker semivariogram,station_holdout,rmse,0.7987777234034086,0.0234181251094234,0.0052364519650265,0,4.85,20\r\nLMC block MLE,station_holdout,rmse,0.7920576827020211,0.0216581505695617,0.0048429096940465,15,1.25,20\r\nLMC semivariogram,station_holdout,rmse,0.7938457803926827,0.0222157324675347,0.0049675887967356,3,2.4,20\r\nPCA MLE,station_holdout,rmse,0.7944914273429933,0.021858855318194,0.0048877886401814,2,2.65,20\r\nPCA semivariogram,station_holdout,rmse,0.7974396087955962,0.0229499927541,0.0051317743881295,0,3.85,20\r\nKronecker semivariogram,station_holdout,crps,0.4341778853940933,0.0084437823835769,0.0018880871396893,0,5.0,20\r\nLMC block MLE,station_holdout,crps,0.4297443812391529,0.0075067234516871,0.0016785543926264,15,1.25,20\r\nLMC semivariogram,station_holdout,crps,0.4306890322240188,0.0076269065855393,0.0017054281583306,4,2.05,20\r\nPCA MLE,station_holdout,crps,0.4319299849709981,0.0079552513320355,0.0017788482756527,1,2.9,20\r\nPCA semivariogram,station_holdout,crps,0.4333432691704048,0.0082666732811248,0.0018484843404376,0,3.8,20\r\nKronecker semivariogram,period_holdout,rmse,0.3740718140059438,0.2152922351575666,,0,3.7777777777777777,9\r\nLMC block MLE,period_holdout,rmse,0.3679647693405867,0.2117688349499294,,9,1.0,9\r\nLMC semivariogram,period_holdout,rmse,0.3742550195330232,0.2135235017158253,,0,3.7777777777777777,9\r\nPCA MLE,period_holdout,rmse,0.3714741830519626,0.2133119185542033,,0,2.111111111111111,9\r\nPCA semivariogram,period_holdout,rmse,0.377877242608931,0.2135820001253603,,0,4.333333333333333,9\r\nKronecker semivariogram,period_holdout,crps,0.2060858939099696,0.1239186383239534,,0,3.333333333333333,9\r\nLMC block MLE,period_holdout,crps,0.2028995262323752,0.1219700482552689,,7,1.3333333333333333,9\r\nLMC semivariogram,period_holdout,crps,0.2084932353522748,0.1227288374485816,,0,4.444444444444445,9\r\nPCA MLE,period_holdout,crps,0.204499031692815,0.1227632792402864,,2,1.7777777777777777,9\r\nPCA semivariogram,period_holdout,crps,0.2078260409981498,0.1229882726688062,,0,4.111111111111111,9\r\n'))
display(pd.DataFrame([{'models_file': CACHE.name,
    'recorded_models_sha256': RECORDED_MODELS_SHA256,
    'distributed_models_sha256': DISTRIBUTED_MODELS_SHA256,
    'current_file_matches_distributed_fit': hashlib.sha256(CACHE.read_bytes()).hexdigest() == DISTRIBUTED_MODELS_SHA256,
    'events': 41, 'observations': 49995}]))
display(recorded_summary)


,models_file,recorded_models_sha256,distributed_models_sha256,current_file_matches_distributed_fit,events,observations
0,models_complete.pkl,49a2a49147ad808ea375805a032d58a20bee5aa84a94fc...,773a73711e8112e28c7d0d05111def6c44456325906cda...,True,41,49995


,method,wls,log_likelihood,pseudo_rmse,pseudo_crps,station_rmse,station_crps,period_rmse,period_crps
0,Kronecker semivariogram,0.230155,-27347.170596,0.371544,0.183786,0.799104,0.434178,0.425594,0.206086
1,PCA MLE,0.364385,-25005.973824,0.368059,0.181674,0.794777,0.431930,0.422421,0.204499
2,PCA semivariogram,0.199046,-25992.745889,0.370698,0.183489,0.797753,0.433343,0.428182,0.207826
3,LMC block MLE,0.160421,-24260.028171,0.361642,0.178070,0.792339,0.429744,0.418642,0.202900
4,LMC semivariogram,0.094248,-25248.894036,0.366163,0.181211,0.794141,0.430689,0.424963,0.208493


In [3]:
display(recorded_statistics.pivot(index='method', columns=['task', 'metric'], values='wins'))
display(recorded_statistics[['method', 'task', 'metric', 'mean', 'sd', 'n']])

task                    random_holdout      station_holdout       \
metric                            rmse crps            rmse crps   
method                                                             
Kronecker semivariogram              0    0               0    0   
LMC block MLE                       20   20              15   15   
LMC semivariogram                    0    0               3    4   
PCA MLE                              0    0               2    1   
PCA semivariogram                    0    0               0    0   

task                    period_holdout       
metric                            rmse crps  
method                                       
Kronecker semivariogram              0    0  
LMC block MLE                        9    7  
LMC semivariogram                    0    0  
PCA MLE                              0    2  
PCA semivariogram                    0    0

,method,task,metric,mean,sd,n
0,Kronecker semivariogram,random_holdout,rmse,0.371522,0.004099,20
1,LMC block MLE,random_holdout,rmse,0.361617,0.004311,20
2,LMC semivariogram,random_holdout,rmse,0.366135,0.004637,20
3,PCA MLE,random_holdout,rmse,0.368036,0.004202,20
4,PCA semivariogram,random_holdout,rmse,0.370675,0.004306,20
5,Kronecker semivariogram,random_holdout,crps,0.183786,0.002141,20
6,LMC block MLE,random_holdout,crps,0.178070,0.002061,20
7,LMC semivariogram,random_holdout,crps,0.181211,0.002110,20
8,PCA MLE,random_holdout,crps,0.181674,0.002067,20
9,PCA semivariogram,random_holdout,crps,0.183489,0.002126,20


## Run a complete real event

This cell actually recomputes all models on the smallest whole event, with 20 entry holdouts, 20 station holdouts, and each available period held out in turn. Masks are generated over the entire dataset before selecting this event, so they match the full evaluation exactly. The displayed likelihood and prediction scores are for this event only; WLS is a fit diagnostic on the entire fitted dataset.

In [4]:
event_id = smallest_event(CACHE, DATASET)
with threadpool_limits(limits=1):
    example = evaluate(CACHE, dataset=DATASET, event_ids=[event_id])
print(f'Live evaluation: event {event_id}')
display(example['run'])
display(example['summary'])

Live evaluation: event 1001


,dataset,seed,repeats,events,observations,mask_sha256,models_sha256,smoke,whole_dataset
0,complete,0,20,1,351,a0a896f6e89b4dc4d5363b2b01fcef1beb3f785a2046a4...,773a73711e8112e28c7d0d05111def6c44456325906cda...,False,False


,method,wls,log_likelihood,pseudo_rmse,pseudo_crps,station_rmse,station_crps,period_rmse,period_crps
0,Kronecker semivariogram,0.230155,-189.003451,0.351522,0.175581,1.093828,0.603576,0.377329,0.187318
1,PCA MLE,0.364385,-182.082139,0.350154,0.175878,1.061330,0.587554,0.375544,0.186616
2,PCA semivariogram,0.199046,-182.888501,0.351343,0.175863,1.082553,0.597310,0.379118,0.187816
3,LMC block MLE,0.160421,-178.363803,0.350971,0.175936,1.031303,0.566156,0.374615,0.185869
4,LMC semivariogram,0.094248,-189.063023,0.360819,0.181297,1.033018,0.565916,0.384412,0.194037


## Rerun every event

Set `RUN_ALL_EVENTS = True` and execute this cell to rebuild the full table and replicate statistics from the model PKL. This is computationally expensive because each event requires a joint covariance factorization for each model. The results remain in memory and are displayed directly below. The two input PKLs are not modified.

In [5]:
RUN_ALL_EVENTS = False
if RUN_ALL_EVENTS:
    with threadpool_limits(limits=1):
        full_run = evaluate(CACHE, dataset=DATASET, verbose=True)
    display(full_run['summary'])
    display(full_run['replicate_statistics'])
else:
    print('Full rerun not requested. The recorded all-event table and live single-event example are shown above.')

Full rerun not requested. The recorded all-event table and live single-event example are shown above.


## Inspect individual predictions or export a table

`example` (or `full_run`) contains `event_likelihood`, `random_holdout`, `station_holdout`, `period_holdout`, `summary`, `replicate_statistics`, and `run` DataFrames. Gaussian conditioning is implemented by `ConditionalCache` in `evaluate.py`. To export a table when needed, call its standard `to_csv` method.

In [6]:
display(example['random_holdout'][['method', 'eqid', 'repeat', 'n_observations', 'sse', 'crps_sum']].head())
# Optional: example['summary'].to_csv('event_summary.csv', index=False)

,method,eqid,repeat,n_observations,sse,crps_sum
0,Kronecker semivariogram,1001,0,35,3.046629,4.873875
1,Kronecker semivariogram,1001,1,35,6.056620,7.342470
2,Kronecker semivariogram,1001,2,35,3.798188,5.758173
3,Kronecker semivariogram,1001,3,35,4.746766,6.385029
4,Kronecker semivariogram,1001,4,35,4.014356,5.921054
